# Session 5 - Reconstruct, Verify, Ship
## AI Productization and Final Demo

> **A demo is not evidence until it survives a restart.**

The same evidence chain built earlier is carried into a runnable application:

**DATA CONTRACT -> LEGITIMATE MODEL -> EXPLANATION -> RELIABILITY ROUTER -> ARTIFACT BUNDLE -> APP -> DEPLOYMENT**

### Final target
A reproducible **Reliable AI Risk Intelligence** application will be rebuilt from source and persisted artifacts, verified in a new process, exercised through all four Session 4 routes, and made ready for local or internet deployment.

### Fixed boundary
- No new model shopping is performed.
- No new threshold is tuned from 2025.
- The 2025 holdout remains final evaluation evidence.
- Productization changes packaging and verification, not the scientific decision rule.


## 0. Course continuity - what is being carried into Session 5?

| Earlier work | Evidence retained | Session 5 consequence |
|---|---|---|
| Session 1 - data fusion | 840-row modeling contract | the application uses the same engineered schema |
| Session 2 - leakage-aware selection | `logistic` is the best eligible path; the leaky candidate is ineligible | the selected object is persisted, not a favorite model name |
| Session 3 - explainability/debugging | global reliance + local evidence + error-case reasoning | accepted decisions can carry explanation evidence |
| Session 4 - reliable AI | calibration + contract + anomaly + confidence routing | the four route states become the application contract |
| **Session 5 - productization** | source + artifacts + tests + app | the behavior must survive restart and deployment |

**Reference checkpoints:** 840 rows, 16 model features, `logistic`, 2025 Macro F1 about 0.6357, temperature about 1.0883, review threshold 0.650, anomaly threshold about -0.0404.


## Live build format

Seven short engineering challenges are used. Every challenge follows the same pattern:

**PREDICT -> BUILD/BREAK -> RUN -> INSPECT -> DEFEND**

| Round | Challenge |
|---|---|
| 0 | What disappears after a restart? |
| 1 | Which files are actually part of the product? |
| 2 | Can metadata corruption be detected? |
| 3 | Can all four Session 4 routes be reproduced after loading? |
| 4 | What must a human-review record contain? |
| 5 | Which regression test catches which failure? |
| 6 | Can a clean process verify the full product? |
| 7 | Which deployment path fits the goal: local, temporary public, or hosted? |

A wrong prediction is useful when the reasoning behind it can be defended and then corrected by evidence.


## 1. Find the course package and import the shared runtime

The notebook is not allowed to become a second implementation. The `src/` package remains the source of truth.


In [1]:
import json
import shutil
import subprocess
import sys
import tempfile
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

current = Path.cwd().resolve()
for candidate in [current, *current.parents]:
    if (candidate / "src" / "reliable_ai").exists():
        ROOT = candidate
        break
else:
    raise FileNotFoundError("A project root containing src/reliable_ai could not be found.")

SRC_DIR = ROOT / "src"
DATA_DIR = SRC_DIR / "data"
MODEL_DIR = SRC_DIR / "model"
REPORTS_DIR = SRC_DIR / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from reliable_ai import (
    CLASS_NAMES,
    LEGITIMATE_MODEL_NAMES,
    MODEL_FEATURES,
    POLICY_VERSION,
    REQUIRED_ARTIFACT_FILES,
    encode_target,
    fit_and_persist_artifacts,
    load_artifacts,
    run_training_comparison,
    validate_artifact_contract,
)
from app import get_app, predict_request
from verify_product import DEMO_CASES, EXPECTED

print("Project root:", ROOT)
print("Source path :", SRC_DIR)
print("Class order :", list(CLASS_NAMES))
print("Features    :", len(MODEL_FEATURES))


Project root: A:\BRAND NEW START\Ongoing\Summer26\Advanced_AI_Practical
Source path : A:\BRAND NEW START\Ongoing\Summer26\Advanced_AI_Practical\src
Class order : ['LOW', 'MEDIUM', 'HIGH']
Features    : 16


### ROUND 0 - What disappears after a restart?

A notebook has just produced a correct decision. The kernel is restarted.

Which item is most likely to disappear while the files remain intact?

**A.** `metadata.json`  
**B.** an already-fitted Python object stored only in memory  
**C.** `risk_model.joblib`  
**D.** `threshold_analysis.csv`

Lock one choice and one reason before the answer is revealed.


**Reference:** B. A fresh process cannot inherit an estimator, dataframe, threshold, or path that existed only in notebook memory.


In [2]:
round0_choice = "B"
round0_reason = "A fresh process cannot inherit Python objects that were never persisted."
print("Locked:", round0_choice, "|", round0_reason)


Locked: B | A fresh process cannot inherit Python objects that were never persisted.


## 2. Rebuild the evidence-selected path from the established data contract

The candidate registry is rerun only to reconstruct the already-defined training path. Eligibility remains the first gate.


In [3]:
comparison = run_training_comparison(DATA_DIR)
metrics = comparison["metrics"].copy()
metrics["eligible"] = metrics["model"].isin(LEGITIMATE_MODEL_NAMES)

selected_name = comparison["selected_model_name"]
selected_model = comparison["selected_model"]
selected_features = list(comparison["selected_features"])
split = comparison["split"]
train, calibration, test = split["train"], split["calibration"], split["test"]

view = metrics.sort_values(["eligible", "macro_f1", "log_loss"], ascending=[False, False, True]).reset_index(drop=True)
display(view)
print("Selected eligible model:", selected_name)
print(f"Split: train={len(train)} | calibration={len(calibration)} | test={len(test)}")

assert len(train) == 600 and len(calibration) == 120 and len(test) == 120
assert selected_name == "logistic"
assert selected_features == list(MODEL_FEATURES)
assert metrics.loc[metrics["model"] == "advanced_leaky", "macro_f1"].iloc[0] > metrics.loc[metrics["model"] == selected_name, "macro_f1"].iloc[0]


,model,accuracy,balanced_accuracy,macro_f1,log_loss,eligible
0,logistic,0.7417,0.7418,0.6357,0.5700,True
1,advanced_correct,0.7000,0.5030,0.4703,0.9523,True
2,dummy,0.4083,0.3333,0.1933,21.3258,True
3,advanced_leaky,0.9500,0.9706,0.9654,0.2316,False


Selected eligible model: logistic
Split: train=600 | calibration=120 | test=120


### Evidence reading

The high leaky score is deliberately visible. It is rejected because the feature contract makes it ineligible, not because its score is small. This is the direct bridge from Session 2 into productization.


## 3. Rebuild and persist the Session 4 reliability contract

The shared `src` implementation is now used to persist the **selected model + calibration + anomaly model + reference medians + policy evidence + metadata + SHA-256 manifest** as one coherent bundle.


In [4]:
persisted = fit_and_persist_artifacts(comparison, DATA_DIR, MODEL_DIR)
artifacts = load_artifacts(MODEL_DIR, verify=True)
meta = artifacts["metadata"]

checkpoint = pd.DataFrame([
    ["selected model", meta["selected_model_name"]],
    ["2025 Macro F1", meta["selected_test_metrics"]["macro_f1"]],
    ["temperature", meta["temperature"]],
    ["review threshold", meta["review_threshold"]],
    ["anomaly threshold", meta["anomaly_threshold"]],
    ["policy version", meta["policy_version"]],
], columns=["checkpoint", "value"])
display(checkpoint)


# What does Assert?
assert meta["selected_model_name"] == "logistic"
assert abs(meta["selected_test_metrics"]["macro_f1"] - 0.6357) < 1e-4
assert abs(meta["temperature"] - 1.0883) < 0.02
assert abs(meta["review_threshold"] - 0.650) < 1e-12
assert abs(meta["anomaly_threshold"] - (-0.040398)) < 0.01
assert meta["policy_version"] == POLICY_VERSION



,checkpoint,value
0,selected model,logistic
1,2025 Macro F1,0.6357
2,temperature,1.088331
3,review threshold,0.65
4,anomaly threshold,-0.040398
5,policy version,reliability_policy_v2


## 4. Report the frozen policy on 2025 - do not tune it

The final holdout is used only to report behavior under the already-selected model and already-selected policy.


In [5]:
X_test = test[MODEL_FEATURES]
y_test = encode_target(test["risk_label"])
raw = artifacts["model"].predict_proba(X_test) # predict_proba vs predict?
calibrated = artifacts["temperature_scaler"].transform(raw)
pred = calibrated.argmax(axis=1)
conf = calibrated.max(axis=1)
accepted = conf >= artifacts["review_threshold"] # If confidence is higher than Taw, accept.

from sklearn.metrics import f1_score
macro_f1 = float(f1_score(y_test, pred, average="macro"))
coverage = float(accepted.mean())
review_rate = float(1 - coverage)
selective_accuracy = float((pred[accepted] == y_test[accepted]).mean())

display(pd.DataFrame({
    "metric": ["macro_f1", "coverage", "review_rate", "selective_accuracy"],
    "2025": [macro_f1, coverage, review_rate, selective_accuracy],
}))
print("The test year is being measured here, not optimized.")


,metric,2025
0,macro_f1,0.635676
1,coverage,0.708333
2,review_rate,0.291667
3,selective_accuracy,0.800000


The test year is being measured here, not optimized.


### ROUND 1 - Artifact detective

Classify the files below into three groups:

- **runtime/reconstruction** - required to reproduce behavior;
- **decision evidence** - not called for every request, but needed to defend the product;
- **incidental** - useful perhaps, but not part of this product contract.

Candidates: `risk_model.joblib`, `temperature_scaler.joblib`, `anomaly_model.joblib`, `reference_medians.joblib`, `metadata.json`, `model_comparison.csv`, `threshold_analysis.csv`, `screenshot.png`, `notes.txt`.


**Reference:** the four joblib objects plus `metadata.json` reconstruct runtime behavior; `model_comparison.csv` and `threshold_analysis.csv` defend selection/policy; screenshots and notes do not define the executable contract.


In [6]:
runtime = ["risk_model.joblib", "temperature_scaler.joblib", "anomaly_model.joblib", "reference_medians.joblib", "metadata.json"]
evidence = ["model_comparison.csv", "threshold_analysis.csv"]
incidental = ["screenshot.png", "notes.txt"]
print("RUNTIME:", runtime)
print("EVIDENCE:", evidence)
print("INCIDENTAL:", incidental)


RUNTIME: ['risk_model.joblib', 'temperature_scaler.joblib', 'anomaly_model.joblib', 'reference_medians.joblib', 'metadata.json']
EVIDENCE: ['model_comparison.csv', 'threshold_analysis.csv']
INCIDENTAL: ['screenshot.png', 'notes.txt']


## 5. Inspect the persisted artifact contract

The manifest is a byte-level identity check. It answers **which files were deployed**, not **whether the scientific model is good**.


In [7]:
contract = validate_artifact_contract(MODEL_DIR)
assert contract["ok"], contract["problems"]
manifest_rows = []
for name, info in contract["manifest"]["files"].items():
    manifest_rows.append({"artifact": name, "bytes": info["bytes"], "sha256_prefix": info["sha256"][:16]})
display(pd.DataFrame(manifest_rows))
print("Artifact contract: PASSED")


,artifact,bytes,sha256_prefix
0,risk_model.joblib,4666,d817c13828d84c0a
1,temperature_scaler.joblib,90,b297a2e39a6476af
2,anomaly_model.joblib,3557766,fe4130ee17cb8374
3,reference_medians.joblib,2088,fcab7ab6369d6d0a
4,metadata.json,1467,07c5a4c2a0a0a1ea
5,model_comparison.csv,208,8e7d97c937da8193
6,threshold_analysis.csv,548,eff1b1202fac8858


Artifact contract: PASSED


### ROUND 2 - Contract sabotage

A copy of `metadata.json` will be changed while the model bytes remain untouched.

Predict the result before execution:

1. Can Python still read the JSON?
2. Is it still the same declared product?
3. Which check should fail first: feature-order identity or model accuracy?


**Reference:** Python can read valid-but-corrupted JSON, but the product identity is broken. Contract/fingerprint validation must fail before any accuracy discussion.


In [8]:
python_can_read = "yes"
same_product = "no"
first_failure = "artifact/feature contract"
print({"python_can_read": python_can_read, "same_product": same_product, "first_failure": first_failure})


{'python_can_read': 'yes', 'same_product': 'no', 'first_failure': 'artifact/feature contract'}


## 6. Break a copied bundle safely, then prove that the validator catches it

The real `src/model/` directory is not modified. A temporary copy is sabotaged.


In [9]:
with tempfile.TemporaryDirectory() as tmp:
    broken_dir = Path(tmp) / "model"
    shutil.copytree(MODEL_DIR, broken_dir)
    meta_path = broken_dir / "metadata.json"
    broken_meta = json.loads(meta_path.read_text(encoding="utf-8"))
    broken_meta["model_features"] = list(reversed(broken_meta["model_features"]))
    meta_path.write_text(json.dumps(broken_meta, indent=2), encoding="utf-8")

    broken = validate_artifact_contract(broken_dir)
    print("Contract OK:", broken["ok"])
    print("Detected problems:")
    for issue in broken["problems"]:
        print(" -", issue)
    assert not broken["ok"]
    assert any("model_features" in issue or "fingerprint mismatch" in issue for issue in broken["problems"])

print("The original bundle remains valid:", validate_artifact_contract(MODEL_DIR)["ok"])


Contract OK: False
Detected problems:
 - metadata model_features do not match the approved feature order
 - fingerprint mismatch: metadata.json
The original bundle remains valid: True


### ROUND 3 - Route arena

Four requests from Session 4 are loaded below. Predict the route before the router is executed:

- `ACCEPT_CASE`
- `LOW_CONFIDENCE_CASE`
- `ANOMALY_CASE`
- `INVALID_CASE`

Allowed states: `ACCEPT_AI_DECISION`, `HUMAN_REVIEW_LOW_CONFIDENCE`, `HUMAN_REVIEW_ANOMALY`, `REJECT_INVALID_INPUT`.


**Reference:** the names describe the intended deterministic test cases, but the evidence is still produced by the loaded router below.


In [10]:
route_prediction = dict(EXPECTED)
route_prediction


{'ACCEPT_CASE': 'ACCEPT_AI_DECISION',
 'LOW_CONFIDENCE_CASE': 'HUMAN_REVIEW_LOW_CONFIDENCE',
 'ANOMALY_CASE': 'HUMAN_REVIEW_ANOMALY',
 'INVALID_CASE': 'REJECT_INVALID_INPUT'}

## 7. Reproduce all four Session 4 routes from loaded artifacts

The UI is bypassed here. The shared application boundary is called directly.


In [11]:
artifacts = load_artifacts(MODEL_DIR, verify=True)
route_results = {}
rows = []
for name, request in DEMO_CASES.items():
    result = predict_request(request, artifacts)
    route_results[name] = result
    assert result["status"] == EXPECTED[name], (name, result)
    rows.append({
        "case": name,
        "status": result["status"],
        "reason": result["reason"],
        "prediction": result["prediction"],
        "confidence": result["confidence"],
        "anomaly_score": result["anomaly_score"],
        "problems": len(result["problems"]),
    })
display(pd.DataFrame(rows))
print("All four routes were reproduced from the persisted bundle.")


,case,status,reason,prediction,confidence,anomaly_score,problems
0,ACCEPT_CASE,ACCEPT_AI_DECISION,valid_familiar_and_confident,LOW,0.997787,-0.017264,0
1,LOW_CONFIDENCE_CASE,HUMAN_REVIEW_LOW_CONFIDENCE,confidence_below_declared_policy,LOW,0.554831,0.004864,0
2,ANOMALY_CASE,HUMAN_REVIEW_ANOMALY,valid_but_out_of_distribution,None,NaN,-0.078125,0
3,INVALID_CASE,REJECT_INVALID_INPUT,input_contract_failed,None,NaN,NaN,11


All four routes were reproduced from the persisted bundle.


### Route semantics

- `REJECT_INVALID_INPUT` = hard input contract failure; no prediction is produced.
- `HUMAN_REVIEW_ANOMALY` = valid but unfamiliar input.
- `HUMAN_REVIEW_LOW_CONFIDENCE` = valid and familiar, but below the automatic-action confidence rule.
- `ACCEPT_AI_DECISION` = every earlier gate passed; acceptance is not certainty.


### ROUND 4 - Human-review ticket

A reviewer should not need to reopen the notebook to understand why a case was deferred.

Choose the minimum fields that must travel with a review case. Include enough information to identify the route, thresholds, model/policy version, and useful explanation evidence.


**Reference:** status, reason, prediction, confidence, review/anomaly thresholds and score, model/policy versions, explanation/top factors, and a timestamp form a compact defensible record.


In [12]:
review_fields = [
    "timestamp_utc", "status", "reason", "prediction", "confidence",
    "review_threshold", "anomaly_score", "anomaly_threshold",
    "model_version", "policy_version", "top_factors",
]
print("Reference fields:", review_fields)


Reference fields: ['timestamp_utc', 'status', 'reason', 'prediction', 'confidence', 'review_threshold', 'anomaly_score', 'anomaly_threshold', 'model_version', 'policy_version', 'top_factors']


## 8. Export and verify one human-review JSON record

Only `HUMAN_REVIEW_*` outcomes are accepted by this helper.


In [13]:
def export_review_record(result: dict, path: Path) -> dict:
    if not str(result.get("status", "")).startswith("HUMAN_REVIEW_"):
        raise ValueError("Only HUMAN_REVIEW_* outcomes belong in this review queue.")
    factors = []
    for item in (result.get("explanation") or [])[:5]:
        if isinstance(item, dict) and item.get("feature"):
            factors.append({"feature": item["feature"], "effect": float(item["effect"])})
        elif isinstance(item, str):
            factors.append(item)
    record = {
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "status": result["status"],
        "reason": result["reason"],
        "prediction": result["prediction"],
        "confidence": result["confidence"],
        "review_threshold": result["review_threshold"],
        "anomaly_score": result["anomaly_score"],
        "anomaly_threshold": result["anomaly_threshold"],
        "model_version": result["model_version"],
        "policy_version": result["policy_version"],
        "top_factors": factors,
    }
    path.write_text(json.dumps(record, indent=2), encoding="utf-8")
    reloaded = json.loads(path.read_text(encoding="utf-8"))
    assert reloaded["status"] == record["status"]
    return reloaded

review_path = REPORTS_DIR / "session5_review_record.json"
review_record = export_review_record(route_results["LOW_CONFIDENCE_CASE"], review_path)
print(json.dumps(review_record, indent=2))


{
  "timestamp_utc": "2026-08-08T10:12:11.221197+00:00",
  "status": "HUMAN_REVIEW_LOW_CONFIDENCE",
  "reason": "confidence_below_declared_policy",
  "prediction": "LOW",
  "confidence": 0.5548305281955214,
  "review_threshold": 0.65,
  "anomaly_score": 0.004864015539702116,
  "anomaly_threshold": -0.04039784609454056,
  "model_version": "logistic",
  "policy_version": "reliability_policy_v2",
  "top_factors": [
    "Confidence 0.5548 below threshold 0.65"
  ]
}


## 9. Keep the interface thin

The application is allowed to collect inputs and render evidence. It is **not** allowed to engineer a second feature path, fit calibration, choose thresholds, or recreate routing logic.

`UI INPUT -> predict_request() -> make_safe_decision() -> structured evidence -> UI`


In [15]:
app_path = SRC_DIR / "app.py"
app_text = app_path.read_text(encoding="utf-8")
checks = {
    "shared decision function used": "make_safe_decision" in app_text,
    "deployment flag available": "--share" in app_text,
    "artifact verification used": "load_artifacts" in app_text,
}
print(checks)
assert all(checks.values())

demo = get_app()
print("Gradio app object:", type(demo).__name__)
assert demo is not None


{'shared decision function used': True, 'deployment flag available': True, 'artifact verification used': True}
Gradio app object: Interface


### ROUND 5 - Regression-test detective

Match each failure to the strongest regression check:

**A.** feature order in metadata was reversed  
**B.** calibration data leaked into the test year  
**C.** `advanced_leaky` became selectable  
**D.** calibrated probabilities no longer sum to one  
**E.** a model file was replaced without updating the manifest

Checks:
1. chronological split invariant  
2. prohibited-candidate selection test  
3. probability-distribution invariant  
4. artifact contract / fingerprint test  
5. metadata feature-schema test


**Reference:** A->5, B->1, C->2, D->3, E->4. A green suite protects declared behavior; it does not guarantee future accuracy or causality.


In [16]:
test_match = {"A": 5, "B": 1, "C": 2, "D": 3, "E": 4}
print(test_match)


{'A': 5, 'B': 1, 'C': 2, 'D': 3, 'E': 4}


## 10. Execute the regression contract

The suite includes the earlier data/model/reliability checks plus Session 5 manifest tampering and audit-field checks.


In [17]:
pytest_proc = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", str(SRC_DIR / "tests" / "test_system.py")],
    capture_output=True,
    text=True,
    cwd=str(SRC_DIR),
)
print(pytest_proc.stdout)
if pytest_proc.stderr:
    print(pytest_proc.stderr)
assert pytest_proc.returncode == 0


..........                                                               [100%]
============================== warnings summary ===============================
tests/test_system.py::test_leaky_model_is_not_eligible_for_selection
tests/test_system.py::test_persisted_model_matches_selected_model_and_runs
tests/test_system.py::test_invalid_input_is_rejected
tests/test_system.py::test_artifact_manifest_and_contract_are_consistent
tests/test_system.py::test_artifact_contract_detects_metadata_tampering
tests/test_system.py::test_decision_result_contains_session4_audit_fields
  a:\anaconda3_23.7.3\envs\mitocluster\Lib\site-packages\sklearn\linear_model\_logistic.py:451: DeprecationWarning: scipy.optimize: The `disp` and `iprint` options of the L-BFGS-B solver are deprecated and will be removed in SciPy 1.18.0.
    opt_res = optimize.minimize(

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
10 passed, 6 warnings in 8.61s



### ROUND 6 - Final boss: cold-start verification

Before execution, name one hidden dependency that a notebook `Run All` could accidentally provide but a new process cannot.


**Reference:** examples include an in-memory estimator, threshold, dataframe, modified import path, or working-directory assumption. A new process must reconstruct everything from explicit source and files.


In [18]:
hidden_dependency = "an in-memory fitted estimator, threshold, or dataframe"
why_new_process_is_stronger = "A clean process cannot inherit notebook state."
print(hidden_dependency, "|", why_new_process_is_stronger)


an in-memory fitted estimator, threshold, or dataframe | A clean process cannot inherit notebook state.


## 11. Run the cold-start product verifier

This command starts a separate Python process, validates fingerprints and metadata, loads the product, and reproduces all four routes.


In [19]:
verify_proc = subprocess.run(
    [sys.executable, str(SRC_DIR / "verify_product.py")],
    capture_output=True,
    text=True,
    cwd=str(SRC_DIR),
)
print(verify_proc.stdout)
if verify_proc.stderr:
    print(verify_proc.stderr)
assert verify_proc.returncode == 0
assert "PRODUCT VERIFICATION PASSED" in verify_proc.stdout


SESSION 5 COLD-START PRODUCT VERIFICATION
[PASS] artifact presence + fingerprints + metadata identity
[PASS] selected model loaded: logistic
[PASS] ACCEPT_CASE          -> ACCEPT_AI_DECISION
[PASS] LOW_CONFIDENCE_CASE  -> HUMAN_REVIEW_LOW_CONFIDENCE
[PASS] ANOMALY_CASE         -> HUMAN_REVIEW_ANOMALY
[PASS] INVALID_CASE         -> REJECT_INVALID_INPUT
PRODUCT VERIFICATION PASSED
Next: python app.py
Temporary public link: python app.py --share



### ROUND 7 - Ship it: choose the deployment path

Three deployment goals are available:

**LOCAL** - fastest and most private; runs only on the current machine/network configuration.  
**TEMPORARY PUBLIC** - `python app.py --share`; useful for a short live demonstration.  
**HOSTED GRADIO SPACE** - upload the `src/` runtime to a Hugging Face Gradio Space for a persistent internet demo under the platform's available free CPU quota.

Choose one path for each scenario:

1. A five-minute classroom link with no cloud setup.
2. A persistent portfolio/demo URL.
3. A disconnected or privacy-sensitive demonstration.


**Reference:** 1->temporary public Gradio share link, 2->Hugging Face Gradio Space, 3->local. Public hosting is a deployment choice, not evidence that the model is scientifically valid.


In [20]:
deployment_choices = {1: "TEMPORARY PUBLIC", 2: "HOSTED GRADIO SPACE", 3: "LOCAL"}
print(deployment_choices)


{1: 'TEMPORARY PUBLIC', 2: 'HOSTED GRADIO SPACE', 3: 'LOCAL'}


## 12. Deployment commands - one codebase, three launch modes

### Local
```bash
cd src
python -m pip install -r requirements.txt
python train_model.py
python verify_product.py
python app.py
```

### Temporary public Gradio link
```bash
cd src
python app.py --share
```
The link depends on Gradio's sharing service and is intended for short demonstrations.

### Hugging Face Spaces
Create a **Gradio** Space and upload the **contents of `src/`** so that `app.py`, `requirements.txt`, `reliable_ai/`, `model/`, and `data/` are at the Space repository root. The app can then be rebuilt by the hosted environment. A free CPU tier may be used when it is available within the account/platform quota.

The detailed checklist is stored in `src/DEPLOYMENT.md`.


In [21]:
required_deploy_files = [
    SRC_DIR / "app.py",
    SRC_DIR / "requirements.txt",
    SRC_DIR / "DEPLOYMENT.md",
    SRC_DIR / "verify_product.py",
    MODEL_DIR / "artifact_manifest.json",
]
for path in required_deploy_files:
    print("OK" if path.exists() else "MISSING", "-", path.relative_to(ROOT))
assert all(path.exists() for path in required_deploy_files)

smoke_proc = subprocess.run(
    [sys.executable, str(SRC_DIR / "app.py"), "--smoke-only", "--no-browser"],
    capture_output=True,
    text=True,
    cwd=str(SRC_DIR),
)
print(smoke_proc.stdout)
assert smoke_proc.returncode == 0


OK - src\app.py
OK - src\requirements.txt
OK - src\DEPLOYMENT.md
OK - src\verify_product.py
OK - src\model\artifact_manifest.json
RELIABLE AI RISK INTELLIGENCE APPLICATION

Loading and validating the persisted artifact bundle...
  Model:             logistic
  Macro F1 (2025):   0.6357
  Temperature:       1.0883
  Review threshold:  0.650
  Anomaly threshold: -0.040398
  Artifact contract: PASSED

Smoke status: ACCEPT_AI_DECISION



## 13. Final evidence record

The application is considered ready for demonstration only when the data/model/policy/software chain is visible in one record.


In [22]:
final_record = {
    "data_rows": 840,
    "model_features": len(MODEL_FEATURES),
    "selected_model": meta["selected_model_name"],
    "future_2025_macro_f1": round(macro_f1, 4),
    "temperature": round(meta["temperature"], 4),
    "review_threshold": round(meta["review_threshold"], 3),
    "anomaly_threshold": round(meta["anomaly_threshold"], 6),
    "coverage_2025": round(coverage, 4),
    "review_rate_2025": round(review_rate, 4),
    "selective_accuracy_2025": round(selective_accuracy, 4),
    "artifact_contract": "PASSED" if validate_artifact_contract(MODEL_DIR)["ok"] else "FAILED",
    "four_routes": "PASSED",
    "cold_start": "PASSED" if verify_proc.returncode == 0 else "FAILED",
    "regression_tests": "PASSED" if pytest_proc.returncode == 0 else "FAILED",
    "deployment_smoke": "PASSED" if smoke_proc.returncode == 0 else "FAILED",
}
final_path = REPORTS_DIR / "session5_final_evidence.json"
final_path.write_text(json.dumps(final_record, indent=2), encoding="utf-8")
print(json.dumps(final_record, indent=2))
print("Saved:", final_path)


{
  "data_rows": 840,
  "model_features": 16,
  "selected_model": "logistic",
  "future_2025_macro_f1": 0.6357,
  "temperature": 1.0883,
  "review_threshold": 0.65,
  "anomaly_threshold": -0.040398,
  "coverage_2025": 0.7083,
  "review_rate_2025": 0.2917,
  "selective_accuracy_2025": 0.8,
  "artifact_contract": "PASSED",
  "four_routes": "PASSED",
  "cold_start": "PASSED",
  "regression_tests": "PASSED",
  "deployment_smoke": "PASSED"
}
Saved: A:\BRAND NEW START\Ongoing\Summer26\Advanced_AI_Practical\src\reports\session5_final_evidence.json


## Final demonstration sequence

1. **DATA:** show the 840-row contract and 16 approved model features.
2. **SELECT:** show `advanced_leaky` scoring higher but remaining ineligible; persist `logistic`.
3. **INSPECT:** connect to one Session 3 explanation and its limitation.
4. **CONTROL:** reproduce ACCEPT / LOW-CONFIDENCE REVIEW / ANOMALY REVIEW / REJECT.
5. **RECONSTRUCT:** validate metadata + fingerprints in a clean process.
6. **VERIFY:** run the regression suite and smoke check.
7. **SHIP:** open the local app, request a temporary public link, or publish the same runtime to a Gradio Space.

> The interface is demonstrated after the evidence chain, not instead of it.


## Technical defense and release checks

A release can be defended when the following questions have concrete answers:

- Why is the leaky 0.96-class score invalid?
- Why must feature order travel with the model?
- Why are `T`, the review threshold, and the anomaly threshold product identity?
- Why is anomaly review different from invalid-input rejection?
- What does a SHA-256 fingerprint prove, and what does it not prove?
- Which test detects stale or tampered artifacts?
- Why is one shared decision function used by the notebook, app, and tests?
- Which risks remain outside the regression suite: future drift, bias, causality, and untested security boundaries?

### Final result
**The controlled AI behavior is portable, testable, auditable, and deployable from the same source package.**


## Assignment 05 - Final AI Product and Defense

Required evidence:

1. runnable package with `requirements.txt` and exact launch commands;
2. persisted model/reliability bundle with a valid `artifact_manifest.json`;
3. all four route states demonstrated;
4. one verified human-review JSON record;
5. at least three additional edge/regression tests, each linked to a specific failure;
6. local or hosted demo path documented;
7. cold-start verification output;
8. concise limitations and AI-assistance disclosure.

A polished interface without reconstructable decision evidence is incomplete.
